# eosframes — Demo Notebook

`eosframes` is a Python library and CLI for manipulating inputs and outputs from the [Ersilia Model Hub](https://github.com/ersilia-os/ersilia).

This notebook walks through all major features using two real model output files:

| File | Model | Description | Rows | Features |
|------|-------|-------------|------|----------|
| `data/example_eos4e40_v1.csv` | [eos4e40](https://github.com/ersilia-os/eos4e40) | Inhibition at 50 µM | 100 | 1 |
| `data/example_eos7m30_v1.csv` | [eos7m30](https://github.com/ersilia-os/eos7m30) | 49 ADMET properties (TDC benchmark) | 100 | 49 |

In [ ]:
import json
import shutil
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd

from eosframes import (
    append_files,
    apply_scaler,
    convert_file,
    dedupe_file,
    fit_scaler,
    hstack,
    is_valid_name,
    make_output_name,
    parse_name,
    read_csv,
    read_h5,
    split_csv,
    stack_files,
    transform_file,
    vstack,
    write_csv,
    write_h5,
)

DATA = Path("..") / "data"
EOS4E40 = DATA / "example_eos4e40_v1.csv"
EOS7M30 = DATA / "example_eos7m30_v1.csv"

WORK = Path(tempfile.mkdtemp(prefix="eosframes_demo_"))
print(f"Scratch dir: {WORK}")

---
## 1. Naming Convention

All output files must follow `[prefix_]<model_id>_<version>.<ext>`. Prefixes (dates, project tags) are optional.

In [ ]:
names = [
    "eos4e40_v1.csv",
    "eos7m30_v2.h5",
    "eos4e40_v1_chunks",
    "260313_gardp_eos4e40_v1.csv",  # date + project prefix
    "example_eos7m30_v1.csv",       # descriptive prefix
    "output.csv",                    # invalid — no model_id
]

pd.DataFrame(
    {"name": n, **parse_name(n)} if parse_name(n) else {"name": n, "model_id": None, "version": None, "extension": None, "name_type": None}
    for n in names
)

In [ ]:
assert is_valid_name("example_eos4e40_v1.csv")
assert not is_valid_name("output.csv")

make_output_name("eos4e40", "v1", "csv"), make_output_name("eos7m30", "v1", "h5")

---
## 2. Reading and Writing Files

`model_id` is extracted automatically from the filename and attached as a DataFrame attribute.

In [ ]:
df4e40 = read_csv(EOS4E40)
print(df4e40.model_id, df4e40.shape)
df4e40.head()

In [ ]:
df7m30 = read_csv(EOS7M30)
print(df7m30.model_id, df7m30.shape)
df7m30[["key", "input", "molecular_weight", "logp", "qed"]].head()

In [ ]:
# H5 round-trip — float32 precision loss is expected
h5_path = WORK / "eos4e40_v1.h5"
write_h5(df4e40, h5_path, dtype=np.float32)
df_back = read_h5(h5_path)

max_diff = np.abs(df_back["inhibition_50um"].values - df4e40["inhibition_50um"].values).max()
print(f"max |diff| from float32 cast: {max_diff:.2e}")
assert max_diff < 1e-5

---
## 3. Splitting into Chunks

In [ ]:
chunks_dir = WORK / "eos4e40_chunks"
n = split_csv(EOS4E40, chunks_dir, chunksize=25)

pd.DataFrame(
    {"file": f.name, "rows": len(pd.read_csv(f))}
    for f in sorted(chunks_dir.iterdir())
)

---
## 4. Converting Between Formats

In [ ]:
h5_out = WORK / "eos7m30_v1.h5"
csv_out = WORK / "eos7m30_v1_restored.csv"
h5_from_chunks = WORK / "eos4e40_from_chunks_v1.h5"

convert_file(EOS7M30, h5_out)            # CSV → H5
convert_file(h5_out, csv_out)            # H5 → CSV
convert_file(chunks_dir, h5_from_chunks) # chunks folder → H5

pd.DataFrame([
    {"conversion": "CSV → H5",     "size_kb": round(h5_out.stat().st_size / 1024, 1)},
    {"conversion": "H5 → CSV",     "rows": len(pd.read_csv(csv_out))},
    {"conversion": "chunks → H5",  "shape": str(read_h5(h5_from_chunks).shape)},
])

---
## 5. Horizontal Stacking (multiple models, same molecules)

In [ ]:
# write_csv enforces the naming convention
p4 = WORK / "eos4e40_v1.csv"
p7 = WORK / "eos7m30_v1.csv"
write_csv(df4e40, p4)
write_csv(df7m30, p7)

stacked_path = WORK / "stacked.csv"
stack_files([p4, p7], stacked_path, suffix=True)

stacked = pd.read_csv(stacked_path)
print(f"shape: {stacked.shape}")
stacked[["key", "inhibition_50um.eos4e40", "molecular_weight.eos7m30", "logp.eos7m30", "qed.eos7m30"]].head()

In [ ]:
# DataFrame API equivalent — no files needed
combined = hstack([df4e40, df7m30])
print(f"shape: {combined.shape}")
combined.filter(regex=r"\.(eos4e40|eos7m30)$").head()

---
## 6. Vertical Appending (same model, multiple batches)

In [ ]:
half = len(df4e40) // 2
p_b1, p_b2 = WORK / "eos4e40_v1_b1.csv", WORK / "eos4e40_v1_b2.csv"
df4e40.iloc[:half].to_csv(p_b1, index=False)
df4e40.iloc[half:].to_csv(p_b2, index=False)

appended_path = WORK / "eos4e40_v1_appended.csv"
append_files([p_b1, p_b2], appended_path)

df_appended = pd.read_csv(appended_path)
assert list(df_appended["key"]) == list(df4e40["key"])
print(f"{len(df4e40)} rows → split → append → {len(df_appended)} rows ✓")

---
## 7. Deduplication

In [ ]:
raw_path = WORK / "eos4e40_v1_raw.csv"
pd.concat([df4e40, df4e40.iloc[:10]], ignore_index=True).to_csv(raw_path, index=False)

before, after = dedupe_file(raw_path, WORK / "eos4e40_v1_deduped.csv")
print(f"{before} rows → dedupe → {after} rows ({before - after} duplicates removed)")

---
## 8. Scaling

Standard scaling (zero mean, unit variance) on the eos7m30 ADMET outputs (49 features).

In [ ]:
# DataFrame API: fit and inspect parameters
params = fit_scaler(df7m30)
print(f"method: {params['method']}  |  fitted: {len(params['columns'])} cols  |  skipped: {params['skipped_columns']}")

cols = ["molecular_weight", "logp", "qed"]
pd.DataFrame([params["parameters"][c] for c in cols], index=cols).round(3)

In [ ]:
# After scaling: mean ≈ 0, std ≈ 1
scaled = apply_scaler(df7m30, params)

pd.DataFrame(
    [{"mean": scaled[c].mean(), "std": scaled[c].std(ddof=0)} for c in cols],
    index=cols,
).round(6)

In [ ]:
# File API: fit + save params, then inspect the JSON
json_path = WORK / "eos7m30_v1_scaler.json"
scaled_path = WORK / "eos7m30_v1_scaled.csv"
transform_file(p7, params=json_path, fit=True, output_path=scaled_path)

t = json.loads(json_path.read_text())
{k: t[k] for k in ("model_id", "version", "n_rows", "fitted_at", "method")}

In [ ]:
# Forward pass: load saved params, no refitting
applied_path = WORK / "eos7m30_v1_applied.csv"
transform_file(p7, params=json_path, output_path=applied_path)

fit_vals = pd.read_csv(scaled_path)["molecular_weight"].values
fwd_vals = pd.read_csv(applied_path)["molecular_weight"].values
assert np.allclose(fit_vals, fwd_vals)
print("fit output == forward-pass output ✓")

---
## 9. Hub Data

Fetch model metadata and column definitions from GitHub (requires network access).

In [ ]:
from eosframes import fetch_metadata, fetch_columns

try:
    meta = fetch_metadata("eos4e40")
    pd.Series({k: meta[k] for k in ("Identifier", "Title", "Task", "Input", "Output Type") if k in meta})
except Exception as e:
    print(f"skipped — network unavailable: {e}")

In [ ]:
try:
    fetch_columns("eos7m30", "v1").head()
except Exception as e:
    print(f"skipped — network unavailable: {e}")

---
## 10. Logging

All operations emit structured log messages. Set `DEBUG` for verbose output.

In [ ]:
import logging
from eosframes import get_logger

logger = get_logger()
logger.setLevel(logging.DEBUG)
read_csv(EOS4E40)
logger.setLevel(logging.INFO)

---
## Clean up

In [ ]:
shutil.rmtree(WORK)